# Top People EDA

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import networkx as nx
from networkx.algorithms import community

from collections import Counter
from itertools import combinations
from IPython.display import display  

from itertools import combinations

In [2]:
df = pd.read_parquet("../data_storage/streamlit_app_data/final_dataset_with_attribution.parquet")

In [4]:
people = pd.read_csv("../data_storage/final_data/persons_detected.parquet")

In [6]:
people_row = pd.read_parquet("../data_storage/streamlit_app_data/persons_by_row_cleaned.parquet")

## People Mentioned

In [7]:
people.head()

,person,count
0,Donald Trump,5807
1,Robert F Kennedy,3817
2,Joe Biden,2110
3,Tedros Ghebreyesus,1254
4,Kamala Harris,564


In [8]:
pd.unique(people.values.ravel())

array(['Donald Trump', 5807, 'Robert F Kennedy', ..., 'William Cornwell',
       'Keri Wiginton', 'Joe Roy'], dtype=object)

## Top People

In [9]:
# Ensure types match
people_row["row_index"] = people_row["row_index"].astype("int64")
df["seq_index"]        = df["seq_index"].astype("int64")

# If persons is a comma-separated string, normalize to list
if people_row["persons"].dtype == "string" or people_row["persons"].dtype == object:
    people_row["persons"] = people_row["persons"].fillna("").apply(
        lambda s: [p.strip() for p in str(s).split(",") if p.strip()]
    )

merged = people_row.merge(
    df,
    left_on="row_index",
    right_on="seq_index",
    how="left"
)

In [10]:
merged.head(5)

,row_index,tag_name_x,persons,has_person,persons_clean,article_body,article_id,author_name,channel_name,circulation_size,...,vipr_score,vipr_weight,people_by_row_clean,headline_token_count,body_token_count,token_count,emotion_body,seq_index,circulation_size_bin,sentiment_score_bin
0,0,public_health,"[Cancer Https, Humanized Mouse]",1,"Brain Cancer, Marketresearch.biz",kabul afghanistan july ani ministry public hea...,19646254249,unknown,web,927,...,19080,1272,"Sharafat Amarakhail, Noor Hussain, Ramazan, Mo...",6,181,187,joy,0.0,1,5
1,0,public_health,"[Cancer Https, Humanized Mouse]",1,"Brain Cancer, Marketresearch.biz",calabria gps face scrutiny prescriptions spark...,19475262153,amelia_romano_cosenza,web,621,...,5832,216,"Amelia Romano, Giovanni Muraca, Roberto Occhiuto",5,311,316,fear,0.0,1,5
2,0,public_health,"[Cancer Https, Humanized Mouse]",1,"Brain Cancer, Marketresearch.biz",begins concerns economy public services politi...,18978544430,unknown,web,904,...,-9450,450,Keir Starmer,7,87,94,fear,0.0,1,2
3,0,public_health,"[Cancer Https, Humanized Mouse]",1,"Brain Cancer, Marketresearch.biz",council passes new smoking rules find find hig...,17924974514,unknown,broadcast,1104,...,-5200,200,"Lloyd Minister, St Paul, Wha, Serger, Sister D...",2,383,385,joy,0.0,2,2
4,1,public_health,"[Elizabeth Warren, Juliann Ventura, Robert Ken...",1,"Elizabeth Warren, Juliann Ventura, Robert F Ke...",juliann ventura hill posted dec cst updated de...,18918986050,juliann_ventura,web,47759,...,9872,617,"Elizabeth Warren, Juliann Ventura, Robert F Ke...",9,329,338,fear,1.0,4,5


In [11]:
person_sentiment = (
    merged.explode("persons")
    .groupby("persons", as_index=False)
    .agg(avg_sent=("sentiment_score", "mean"),
         pos_rate=("sentiment_band", lambda x: (x=="positive").mean()),
         mentions=("article_id", "count"))
    .sort_values("mentions", ascending=False)
)

In [12]:
person_sentiment.head()

,persons,avg_sent,pos_rate,mentions
17721,Donald Trump,-7.197353,0.178072,5290
54419,Robert Kennedy,-6.497396,0.184896,3456
30905,Joe Biden,-7.395758,0.172271,1933
62086,Tedros Ghebreyesus,-7.956294,0.163462,1144
33907,Kamala Harris,-6.301708,0.184061,527


## Top People

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# Get the list of the Top 5 influencers from your existing 'person_sentiment' df
top_5_people = person_sentiment.head(5)['persons'].tolist()

print("Analyzing keywords for:")
print(top_5_people)

Analyzing keywords for:
['Donald Trump', 'Robert Kennedy', 'Joe Biden', 'Tedros Ghebreyesus', 'Kamala Harris']


In [14]:
# Explode the merged df (if not already exploded)
exploded_merged = merged.explode('persons')

# Filter for just the top 5 people
top_influencer_df = exploded_merged[exploded_merged['persons'].isin(top_5_people)]

# *** MODIFIED LINE ***
# Group all the CLEAN keywords into a single string for each person
docs_by_person = top_influencer_df.groupby('persons')['people_by_row_clean'].apply(
    lambda bodies: ' '.join(bodies.fillna(''))
)

print(f"Created {len(docs_by_person)} documents from 'people_by_row_clean'.")

Created 5 documents from 'people_by_row_clean'.


In [15]:
# Start with the default English stop words
custom_stop_words = list(ENGLISH_STOP_WORDS)

# Add the names (and parts of names) of our top influencers
for name in top_5_people:
    parts = name.lower().split()
    custom_stop_words.extend(parts) # Add 'donald', 'trump', 'robert', 'kennedy'
    custom_stop_words.append(name.lower()) # Add 'donald trump', 'robert kennedy'

# Ensure the list is unique
custom_stop_words = list(set(custom_stop_words))

print(f"Custom stop word list now contains {len(custom_stop_words)} words.")

Custom stop word list now contains 333 words.


In [16]:
# Initialize the vectorizer
# We're looking for single words and 2-word phrases (n-grams)
# max_features limits it to the 1000 most common terms overall
vectorizer = TfidfVectorizer(
    stop_words=custom_stop_words,
    ngram_range=(1, 2),  # This gets us 1-word and 2-word phrases
    max_features=1000,
    lowercase=True
)

# Run the TF-IDF analysis
tfidf_matrix = vectorizer.fit_transform(docs_by_person)

# Get the feature names (the keywords)
feature_names = vectorizer.get_feature_names_out()

print("--- Top Keywords & Phrases by Influencer (TF-IDF) ---")

# Loop through each person and print their top keywords
for i, person in enumerate(docs_by_person.index):
    # Get the scores for this person
    scores = tfidf_matrix[i, :].toarray().flatten()
    
    # Get the indices of the top 20 keywords
    top_indices = scores.argsort()[-20:][::-1] # Sort, take top 20, reverse
    
    # Get the actual keyword text
    top_keywords = [feature_names[idx] for idx in top_indices]
    
    print(f"\n## {person.upper()}")
    print(", ".join(top_keywords))

--- Top Keywords & Phrases by Influencer (TF-IDF) ---

## DONALD TRUMP
john, michael, david, james, tom, mark, mike, paul, george, peter, chris, andrew, jennifer, daniel, richard, sarah, anthony, lee, lisa, steve

## JOE BIDEN
john, michael, david, tom, paul, james, andrew, mike, chris, peter, mark, george, et, scott, benjamin, anthony, daniel, brian, richard, sarah

## KAMALA HARRIS
michael, john, david, james, peter, mark, sarah, brian, jennifer, scott, tom, anthony, paul, et, amy, chris, lee, smith, eric, matthew

## ROBERT KENNEDY
john, michael, david, mark, james, tom, paul, mike, peter, chris, george, sarah, andrew, lee, lisa, thomas, jennifer, william, brian, richard

## TEDROS GHEBREYESUS
john, michael, david, james, mike, tom, al, paul, anthony, et, peter, mark, chris, brian, jennifer, sarah, mary, maria, lee, george


In [17]:
# Get the first list of non-useful keywords from our previous output
# THIS LIST IS NOW UPDATED to reflect the noise from 'people_by_row_clean'
generic_topic_words = [
    'tru', 'covid', 'john', 'michael', '19', 'covid 19', 'david', 'jo', 'mark', 
    'ro', 'ghebreyesu', 'br', 'tom', 'adhanom', 'adhanom ghebreyesu', 'james', 
    'johnso', 'mo', 'gr', 'peter', 'paul', 'al', 'harr', 'anthony', 'ho', 'andrew',
    'sarah', 'health', 'public' # Keep a few just in case
]

# Add these new generic words to our existing custom_stop_words
# We'll re-create it from the previous step for clarity
new_stop_words = list(ENGLISH_STOP_WORDS)
for name in top_5_people:
    parts = name.lower().split()
    new_stop_words.extend(parts)
    new_stop_words.append(name.lower())

new_stop_words.extend(generic_topic_words)
new_stop_words = list(set(new_stop_words)) # Ensure uniqueness

print(f"Old stop word count: {len(custom_stop_words)}")
print(f"New, more aggressive stop word count: {len(new_stop_words)}")

Old stop word count: 333
New, more aggressive stop word count: 362


In [18]:
# --- STEP 1: Re-create docs from the full 'article_body' ---
print("Reloading documents from 'article_body'...")
# We use the 'top_influencer_df' you already created
docs_by_person_body = top_influencer_df.groupby('persons')['article_body'].apply(
    lambda bodies: ' '.join(bodies.fillna(''))
)

# --- STEP 2: Use your most aggressive stop-word list (from cell 47) ---
# 'new_stop_words' should still be in memory from when you ran cell 47
print(f"Using aggressive stop list with {len(new_stop_words)} words.")

# --- STEP 3: Run the refined TF-IDF on the full article text ---
vectorizer_final = TfidfVectorizer(
    stop_words=new_stop_words, # Use the BIG list from cell 47
    ngram_range=(1, 2),
    max_features=1000,
    lowercase=True,
    max_df=0.9, 
    min_df=2 
)

# Run the TF-IDF analysis
tfidf_matrix_final = vectorizer_final.fit_transform(docs_by_person_body)

# Get the feature names
feature_names_final = vectorizer_final.get_feature_names_out()

print("\n--- FINAL Top Keywords (from article_body + aggressive filter) ---")

# Loop and print
for i, person in enumerate(docs_by_person_body.index):
    scores = tfidf_matrix_final[i, :].toarray().flatten()
    top_indices = scores.argsort()[-20:][::-1]
    
    top_keywords = [
        feature_names_final[idx] for idx in top_indices if scores[idx] > 0
    ]
    
    print(f"\n## {person.upper()}")
    
    if not top_keywords:
        print("No unique keywords found. Try setting min_df=1 in the code above.")
    else:
        print(", ".join(top_keywords))

Reloading documents from 'article_body'...
Using aggressive stop list with 362 words.

--- FINAL Top Keywords (from article_body + aggressive filter) ---

## DONALD TRUMP
brii, life technology, phi, hbv, frac, auris, endometriosis, betel, aakp, potassium, vmmc, nuzyra, elebsiran, cdc foundation, siga, chikv, proteinuria, jynneos, hbsag, ultra

## JOE BIDEN
stx, vmmc, frac, tmds, carrier screening, zacks, screening market, tdi, rhgs, americorps, aiv, monkeys, life technology, tau, psoriasis, edible mushrooms, brii, glu, bigg, thimerosal

## KAMALA HARRIS
tio, lasv, rhgs, edible mushrooms, brii, mushrooms algae, biofilms, log cfu, violations high, amarin, lap rhgs, admin complaint, arkay, hbv, virology specimen, ladapo, sasakawa, janel, elebsiran, amoeba

## ROBERT KENNEDY
phi, auris, brii, life technology, proteinuria, phi phi, parental, pes, hbv, supplementary, americorps, myopia, immobilization, text text, cdc foundation, grounding, diaper, amanita, left right, plhiv

## TEDROS GHEBRE

In [19]:
def get_top_metadata_by_person(df, column_name):
    """
    Finds the Top 5 most common values for a given column,
    grouped by the top 5 influencers.
    """
    print(f"\n--- Top 5 '{column_name}' by Influencer ---")
    
    # *** THIS IS THE CORRECTED LINE ***
    # It now uses 'df' (the dataframe passed in) 
    # instead of the global 'top_influencer_df'
    grouped_analysis = (
        df.groupby('persons')[column_name]  # <-- This line was the bug
             .value_counts()                # Count occurrences of each value
             .groupby(level=0)              # Group by 'persons' again
             .head(5)                       # Get top 5 for each person
    )
    
    print(grouped_analysis)

In [20]:
# Analyze the top emotions associated with each person
get_top_metadata_by_person(top_influencer_df, 'emotion_body')

# Analyze the top topic tags associated with each person
get_top_metadata_by_person(top_influencer_df, 'tag_name_x')


--- Top 5 'emotion_body' by Influencer ---
persons             emotion_body
Donald Trump        fear            2252
                    joy             1328
                    sadness          661
                    neutral          483
                    anger            387
Joe Biden           fear             803
                    joy              501
                    sadness          252
                    neutral          168
                    anger            139
Kamala Harris       fear             218
                    joy              148
                    sadness           55
                    neutral           54
                    anger             36
Robert Kennedy      fear            1454
                    joy              861
                    sadness          451
                    neutral          297
                    anger            281
Tedros Ghebreyesus  fear             500
                    joy              271
                    s

In [21]:
# --- Filter out 'unknown' authors (Robustly) ---

# We create a new boolean filter
# 1. fillna('') handles any NaN/missing values
# 2. str.lower() handles case (e.g., "Unknown")
# 3. str.strip() handles whitespace (e.g., " unknown ")
is_known_author = top_influencer_df['author_name'].fillna('').str.lower().str.strip() != 'unknown'

# Apply the filter
top_influencer_df_known_authors = top_influencer_df[is_known_author]

print("--- Analyzing by Author (excluding all 'unknown' variations) ---")
# Analyze the top authors associated with each person
get_top_metadata_by_person(top_influencer_df_known_authors, 'author_name')

print("\n\n--- Analyzing by Feed Name ---")
# Analyze the top news feeds associated with each person
get_top_metadata_by_person(top_influencer_df, 'feed_name')

--- Analyzing by Author (excluding all 'unknown' variations) ---

--- Top 5 'author_name' by Influencer ---
persons             author_name      
Donald Trump        mike_stobbe          26
                    meta_time            15
                    brandpoint_bpt       14
                    gabrielle_emanuel    14
                    robin_millard        13
Joe Biden           meta_time             8
                    mike_stobbe           8
                    brandpoint_bpt        7
                    robin_millard         7
                    ari_daniel            4
Kamala Harris       priya_deshmukh        3
                    amanda_seitz          2
                    kilito_chan           2
                    melina_khan           2
                    mike_stobbe           2
Robert Kennedy      mike_stobbe          19
                    gabrielle_emanuel    11
                    brandpoint_bpt       10
                    robin_millard         7
                  